# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 03: Feature Engineering
# ==========================================================

Objective
---------
1. Create the target variable for survival prediction.
2. Select relevant features for machine learning.
3. Encode categorical variables using Spark ML.
4. Assemble features into a feature vector.
5. Split the dataset into training and testing sets.
6. Prepare the final dataset for model training.

Note
----
No data preprocessing.
No model training.
No model evaluation.


In [1]:
# 1. Import Libraries

import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.sql.types import *

In [2]:
# 2. Create Spark Session

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data.loader import (
    create_spark_session,
    load_csv
)

spark = create_spark_session(
    "SEER Breast Cancer Feature Engineering"
)

In [3]:
# 3. Load Clean Dataset

df = load_csv(
    spark,
    "../data/processed/seer_breast_cancer_clean.csv"
)

# Store dataset information

dataset_row_count = df.count()
dataset_column_count = len(df.columns)

In [4]:
# 4. Dataset Overview

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Number of Rows    : {dataset_row_count:,}")
print(f"Number of Columns : {dataset_column_count}")

DATASET OVERVIEW
Number of Rows    : 456,087
Number of Columns : 25


In [5]:
# 5. Schema

print("=" * 60)
print("DATASET SCHEMA")
print("=" * 60)

df.printSchema()

DATASET SCHEMA
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: double (nullable = true)
 |-- Survival_Months: double (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = true)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: double (nullable = true)
 |-- Regional_Nodes_Positive: double (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: integer (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = t

In [6]:
# 6. Preview Dataset

print("=" * 60)
print("DATASET PREVIEW")
print("=" * 60)

df.show(10, truncate=False)

DATASET PREVIEW
+-----------+------+-------------------------+------------------------------+----------+---------------+------------+-----------------------------------+---------+---------+-----------+-----------+-----------------------+-----------------------+----------------+---------------+-------------------------+-----------------------+------+--------------------+--------------------------+-------------------------------------------------------------------------+------------------------------------+------------+----------+
|Age        |Sex   |Race                     |Marital_Status                |Tumor_Size|Survival_Months|Vital_Status|Grade                              |PR_Status|ER_Status|AJCC_T     |AJCC_N     |Regional_Nodes_Examined|Regional_Nodes_Positive|Sequence_Number |Histologic_Type|Laterality               |Diagnostic_Confirmation|AJCC_M|Surgery_Primary_Site|Surgery_Other_Regional    |Surgery_Radiation_Sequence                                               |Radiatio

In [7]:
# 7. Create Target Variable

print("=" * 60)
print("CREATING TARGET VARIABLE")
print("=" * 60)

# Create binary target variable
# Alive = 0
# Dead = 1

from pyspark.sql.functions import when, col

df = df.withColumn(
    "label",
    when(col("Vital_Status") == "Alive", 1).otherwise(0)
)

CREATING TARGET VARIABLE


In [8]:
# 8. Verify Target Variable

print("=" * 60)
print("TARGET VARIABLE VALIDATION")
print("=" * 60)

df.groupBy(
    "Vital_Status",
    "label"
).count() \
.orderBy("label") \
.show(truncate=False)

# df.groupBy("label").count().show()
print()

TARGET VARIABLE VALIDATION
+------------+-----+------+
|Vital_Status|label|count |
+------------+-----+------+
|Dead        |0    |168781|
|Alive       |1    |287306|
+------------+-----+------+




In [9]:
# 9. Feature Selection

print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)

# Features used for machine learning

feature_columns = [
    "Age",
    "Sex",
    "Race",
    "Marital_Status",
    "Tumor_Size",
    "Grade",
    "PR_Status",
    "ER_Status",
    "AJCC_T",
    "AJCC_N",
    "AJCC_M",
    "AJCC_Stage",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive",
    "Histologic_Type",
    "Laterality",
    "Diagnostic_Confirmation",
    "Surgery_Primary_Site",
    "Surgery_Other_Regional",
    "Surgery_Radiation_Sequence",
    "Radiation",
    "Chemotherapy"
]

print(f"Selected Features : {len(feature_columns)}")
print()

for column in feature_columns:
    print(column)

FEATURE SELECTION
Selected Features : 22

Age
Sex
Race
Marital_Status
Tumor_Size
Grade
PR_Status
ER_Status
AJCC_T
AJCC_N
AJCC_M
AJCC_Stage
Regional_Nodes_Examined
Regional_Nodes_Positive
Histologic_Type
Laterality
Diagnostic_Confirmation
Surgery_Primary_Site
Surgery_Other_Regional
Surgery_Radiation_Sequence
Radiation
Chemotherapy


In [10]:
# 10. Histologic Type & Surgery Code Overview
print("Histologic Type & Surgery Code Overview")
print("-"*60)

df.select(
    "Histologic_Type",
    "Surgery_Primary_Site"
).show(20, truncate=False)

df.select(
    "Histologic_Type",
    "Surgery_Primary_Site"
).describe().show()

Histologic Type & Surgery Code Overview
------------------------------------------------------------
+---------------+--------------------+
|Histologic_Type|Surgery_Primary_Site|
+---------------+--------------------+
|8522           |22                  |
|8500           |75                  |
|8520           |51                  |
|8520           |51                  |
|8520           |22                  |
|8500           |41                  |
|8500           |0                   |
|8500           |23                  |
|8500           |0                   |
|8500           |22                  |
|8520           |22                  |
|8500           |22                  |
|8520           |22                  |
|8500           |23                  |
|8500           |22                  |
|8500           |0                   |
|8500           |22                  |
|8500           |44                  |
|8500           |41                  |
|8500           |22                  |
+-

In [11]:
# 11. Handle Missing Numerical Features

print("=" * 60)
print("HANDLE NUMERICAL MISSING VALUES")
print("=" * 60)

numeric_missing_features = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]
for col in numeric_missing_features:
    median_value = df.approxQuantile(col, [0.5], 0.01)[0]

    print(f"{col:<35} Median = {median_value}")

    df = df.fillna({col: median_value})

print()
print("Missing numerical values handled successfully.")

HANDLE NUMERICAL MISSING VALUES
Tumor_Size                          Median = 18.0
Regional_Nodes_Examined             Median = 3.0
Regional_Nodes_Positive             Median = 0.0

Missing numerical values handled successfully.


In [12]:
# 12. Verify Missing Values

print("=" * 60)
print("VERIFY NUMERICAL MISSING VALUES")
print("=" * 60)

for column in numeric_missing_features:
    null_count = (df.filter(F.col(column).isNull()).count())
    print(f"{column:<35} Null = {null_count}")

VERIFY NUMERICAL MISSING VALUES
Tumor_Size                          Null = 0
Regional_Nodes_Examined             Null = 0
Regional_Nodes_Positive             Null = 0


In [13]:
# 13. Feature Metadata

print("="*60)
print("FEATURE METADATA")
print("="*60)

encoded_features = [
    "Age_Vec",
    "Sex_Vec",
    "Race_Vec",
    "Marital_Status_Vec",
    "Grade_Vec",
    "PR_Status_Vec",
    "ER_Status_Vec",
    "AJCC_T_Vec",
    "AJCC_N_Vec",
    "AJCC_M_Vec",
    "AJCC_Stage_Vec",
    "Laterality_Vec",
    "Diagnostic_Confirmation_Vec",
    "Surgery_Other_Regional_Vec",
    "Surgery_Radiation_Sequence_Vec",
    "Radiation_Vec",
    "Chemotherapy_Vec"
]

numeric_features = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive",
    "Histologic_Type",
    "Surgery_Primary_Site"
]

feature_columns = encoded_features + numeric_features

print(f"Encoded Columns      : {len(encoded_features)}")
print(f"Numeric Columns      : {len(numeric_features)}")
print(f"Total Input Columns  : {len(feature_columns)}")

print()

feature_mapping = {
    index: feature
    for index, feature in enumerate(feature_columns)
}

print("Feature Group Mapping")
print("-"*60)

for index, feature in feature_mapping.items():
    print(f"{index:02d} -> {feature}")

FEATURE METADATA
Encoded Columns      : 17
Numeric Columns      : 5
Total Input Columns  : 22

Feature Group Mapping
------------------------------------------------------------
00 -> Age_Vec
01 -> Sex_Vec
02 -> Race_Vec
03 -> Marital_Status_Vec
04 -> Grade_Vec
05 -> PR_Status_Vec
06 -> ER_Status_Vec
07 -> AJCC_T_Vec
08 -> AJCC_N_Vec
09 -> AJCC_M_Vec
10 -> AJCC_Stage_Vec
11 -> Laterality_Vec
12 -> Diagnostic_Confirmation_Vec
13 -> Surgery_Other_Regional_Vec
14 -> Surgery_Radiation_Sequence_Vec
15 -> Radiation_Vec
16 -> Chemotherapy_Vec
17 -> Tumor_Size
18 -> Regional_Nodes_Examined
19 -> Regional_Nodes_Positive
20 -> Histologic_Type
21 -> Surgery_Primary_Site


In [14]:
# 14. Validate Feature Dataset
print("="*60)
print("VALIDATE FEATURE DATASET")
print("="*60)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")
print()

df.select("features","label").show(5, truncate=False)
print()

print("Schema")
print("-"*60)

df.select("features","label").printSchema()
print()

vector_size = len(df.first()["features"])

print(f"Feature Vector Size : {vector_size}")

VALIDATE FEATURE DATASET
Rows    : 456,087
Columns : 26



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `features` cannot be resolved. Did you mean one of the following? [`Race`, `Age`, `Grade`, `label`, `AJCC_M`].;
'Project ['features, label#224]
+- Project [Age#17, Sex#18, Race#19, Marital_Status#20, Tumor_Size#577, Survival_Months#22, Vital_Status#23, Grade#24, PR_Status#25, ER_Status#26, AJCC_T#27, AJCC_N#28, Regional_Nodes_Examined#661, coalesce(nanvl(Regional_Nodes_Positive#30, cast(null as double)), cast(0.0 as double)) AS Regional_Nodes_Positive#745, Sequence_Number#31, Histologic_Type#32, Laterality#33, Diagnostic_Confirmation#34, AJCC_M#35, Surgery_Primary_Site#36, Surgery_Other_Regional#37, Surgery_Radiation_Sequence#38, Radiation#39, Chemotherapy#40, ... 2 more fields]
   +- Project [Age#17, Sex#18, Race#19, Marital_Status#20, Tumor_Size#577, Survival_Months#22, Vital_Status#23, Grade#24, PR_Status#25, ER_Status#26, AJCC_T#27, AJCC_N#28, coalesce(nanvl(Regional_Nodes_Examined#29, c

In [16]:
# 15. Validate Engineered Features

print("=" * 60)
print("VALIDATE ENGINEERED FEATURES")
print("=" * 60)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

print()

print("Preview Dataset")
print("-" * 60)

df.show(5, truncate=False)

print()

print("Schema")
print("-" * 60)

print("Feature Columns")
print("-" * 60)

for c in feature_columns:
    print(c)

VALIDATE ENGINEERED FEATURES
Rows    : 456,087
Columns : 26

Preview Dataset
------------------------------------------------------------
+-----------+------+-----+------------------------------+----------+---------------+------------+-----------------------------------+---------+---------+------+------+-----------------------+-----------------------+----------------+---------------+-------------------------+-----------------------+------+--------------------+--------------------------+-------------------------------------------------------------------------+---------------+------------+----------+-----+
|Age        |Sex   |Race |Marital_Status                |Tumor_Size|Survival_Months|Vital_Status|Grade                              |PR_Status|ER_Status|AJCC_T|AJCC_N|Regional_Nodes_Examined|Regional_Nodes_Positive|Sequence_Number |Histologic_Type|Laterality               |Diagnostic_Confirmation|AJCC_M|Surgery_Primary_Site|Surgery_Other_Regional    |Surgery_Radiation_Sequence         

In [ ]:
# 16. Feature Engineering Report

print("="*60)
print("FEATURE ENGINEERING REPORT")
print("="*60)

print(f"""
      
Target Variable
--------------------------------------------------
label
0 -> Alive
1 -> Dead
      
Feature Selection
--------------------------------------------------
Selected Feature        : {len(feature_columns)}

Categorical Features    : {len(categorical_columns)}

Encoded Features        : {len(encoded_features)}

Numeric Features        : {len(numeric_features)}

Feature Vector
--------------------------------------------------
Output Column           : features

Dataset Split
--------------------------------------------------
Training Records        : {train_df.count():,}

Testing Records         : {test_df.count():,}

Missing Value Treatment
--------------------------------------------------

✓ Target variable created

✓ Categorical features encoded

✓ Numerical features validated

✓ Feature vector assembled

✓ Training and testing datasets generated

""")



Target Variable

--------------------------------------------------
label
0 -> Alive
1 -> Dead

Feature Engineering
--------------------------------------------------
Selected Feature Groups : 22

Categorical Features    : 19

Encoded Features        : 17

Numeric Features        : 5

Feature Vector
--------------------------------------------------
Output Column           : features

Dataset Split
--------------------------------------------------
Training Records        : 365,322

Testing Records         : 90,765

Missing Value Treatment
--------------------------------------------------

Numerical missing values:

- Tumor_Size
- Regional_Nodes_Examined
- Regional_Nodes_Positive

Method                  : Median Imputation




In [ ]:
# 17. Summary

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("""
Notebook 03 Completed Successfully

✓ Target variable created
✓ Feature selection completed
✓ Missing values handled
✓ Categorical features encoded
✓ Feature vector created
✓ Train/Test split completed
✓ Datasets exported

Ready for Notebook 04:
Model Training
""")

FEATURE ENGINEERING SUMMARY
 Feature Engineering Completed Successfully.

Dataset: SEER Breast Cancer

Final Feature Vector: features

Target: label

Data Split
------------------------------------------------------------
Training Dataset    : 365,322

Testing Dataset     : 90,765

Class Imbalance
------------------------------------------------------------
No resampling applied.

Output
------------------------------------------------------------
Feature dataset is ready for:

Notebook 04 — Model Training

Status
------------------------------------------------------------
✓ Feature engineering completed successfully.




